In [2]:
# sudoku_solver.py
# CSP-based Sudoku Solver with Backtracking, Forward Checking, and AC-3

import copy
import time

def read_board(filename):
    """Read Sudoku board from text file"""
    with open(filename, 'r') as f:
        # each line → list of integers
        return [list(map(int, line.split())) for line in f.readlines()]

def print_board(board):
    """Pretty print Sudoku board"""
    for i, row in enumerate(board):
        # print horizontal line after every 3 rows
        if i % 3 == 0 and i != 0:
            print("-" * 21)
        for j, val in enumerate(row):
            # print vertical separator after every 3 columns
            if j % 3 == 0 and j != 0:
                print("|", end=" ")
            # print value or '.' if empty
            print(val if val != 0 else ".", end=" ")
        print()
    print()

class SudokuCSP:
    def __init__(self, board):
        self.size = 9
        # make deep copy of board
        self.board = [row[:] for row in board]
        self.domains = {}  # store domain of each cell
        self.backtrack_count = 0  # count recursive calls
        self.failure_count = 0    # count failures
        self.init_domains()
    
    def init_domains(self):
        """Initialize domains for each cell"""
        for i in range(self.size):
            for j in range(self.size):
                # if empty → domain = {1..9}
                if self.board[i][j] == 0:
                    self.domains[(i, j)] = set(range(1, 10))
                else:
                    # if filled → fixed domain
                    self.domains[(i, j)] = {self.board[i][j]}
        # enforce initial arc consistency
        self.ac3()
    
    def get_peers(self, i, j):
        """Get all peers (same row, column, 3x3 box) for a cell"""
        peers = set()

        # same row
        for col in range(self.size):
            if col != j:
                peers.add((i, col))

        # same column
        for row in range(self.size):
            if row != i:
                peers.add((row, j))

        # same 3x3 box
        box_row, box_col = 3 * (i // 3), 3 * (j // 3)
        for r in range(box_row, box_row + 3):
            for c in range(box_col, box_col + 3):
                if (r, c) != (i, j):
                    peers.add((r, c))

        return peers
    
    def revise(self, x, y):
        """Revise domain of x based on constraint with y"""
        revised = False
        to_remove = set()

        # remove value if no valid value exists in neighbor
        for val_x in self.domains[x]:
            if not any(val_x != val_y for val_y in self.domains[y]):
                to_remove.add(val_x)

        # update domain
        if to_remove:
            self.domains[x] -= to_remove
            revised = True

        return revised
    
    def ac3(self, queue=None):
        """AC-3 algorithm for arc consistency"""
        # initialize queue with all arcs
        if queue is None:
            queue = [(x, y) for x in self.domains for y in self.get_peers(*x)]

        while queue:
            x, y = queue.pop(0)

            # revise domain
            if self.revise(x, y):
                # if domain becomes empty → failure
                if not self.domains[x]:
                    return False

                # add all related arcs again
                for z in self.get_peers(*x):
                    if z != y:
                        queue.append((z, x))
        return True
    
    def forward_check(self, var, value):
        """Forward checking after assigning value to var"""
        # save current domains (for backtracking)
        saved_domains = copy.deepcopy(self.domains)

        # assign value
        self.domains[var] = {value}
        
        # remove value from all peers
        for peer in self.get_peers(*var):
            if value in self.domains[peer]:
                self.domains[peer].discard(value)

                # if domain becomes empty → undo
                if not self.domains[peer]:
                    self.domains = saved_domains
                    return False
        
        # apply AC-3 again after assignment
        if not self.ac3():
            self.domains = saved_domains
            return False
        
        return True
    
    def select_unassigned_variable(self):
        """MRV heuristic - choose variable with smallest domain"""
        min_domain = None
        min_var = None

        for var, dom in self.domains.items():
            # only consider unassigned variables
            if len(dom) > 1:
                # pick variable with minimum remaining values
                if min_domain is None or len(dom) < min_domain:
                    min_domain = len(dom)
                    min_var = var

        return min_var
    
    def is_complete(self):
        """Check if all variables are assigned"""
        # all domains should have exactly 1 value
        return all(len(self.domains[var]) == 1 for var in self.domains)
    
    def backtrack(self):
        """Backtracking search with forward checking"""
        self.backtrack_count += 1
        
        # if complete → solution found
        if self.is_complete():
            return True
        
        var = self.select_unassigned_variable()
        if var is None:
            return True
        
        # try values one by one
        for value in sorted(self.domains[var]):
            if self.forward_check(var, value):
                result = self.backtrack()
                if result:
                    return result
            else:
                # count failure if assignment fails
                self.failure_count += 1
        
        return False
    
    def solve(self):
        """Solve the Sudoku puzzle"""
        success = self.backtrack()

        if success:
            # convert domains → final solution grid
            solution = [[0]*self.size for _ in range(self.size)]
            for (i, j), dom in self.domains.items():
                if len(dom) == 1:
                    solution[i][j] = next(iter(dom))
            return solution

        return None

def solve_sudoku(filename):
    """Solve a Sudoku board from file and print statistics"""
    print(f"\n{'='*50}")
    print(f"Solving: {filename}")
    print(f"{'='*50}")
    
    board = read_board(filename)

    print("Initial board:")
    print_board(board)
    
    csp = SudokuCSP(board)

    start_time = time.time()
    solution = csp.solve()
    end_time = time.time()
    
    if solution:
        print("Solution found:")
        print_board(solution)
    else:
        print("No solution exists!")
    
    print(f"Statistics:")
    print(f"  Backtrack calls: {csp.backtrack_count}")
    print(f"  Failures: {csp.failure_count}")
    print(f"  Time: {(end_time - start_time)*1000:.2f} ms")
    
    return csp.backtrack_count, csp.failure_count

if __name__ == "__main__":
    # list of input files
    boards = ["easy.txt", "medium.txt", "hard.txt", "veryhard.txt"]
    results = {}
    
    for board in boards:
        try:
            bt, fail = solve_sudoku(board)
            results[board] = (bt, fail)
        except FileNotFoundError:
            print(f"Error: {board} not found in current directory!")
            print("Make sure all four board files exist.")
            break
    
    # print summary table
    print(f"\n{'='*50}")
    print("SUMMARY TABLE")
    print(f"{'='*50}")
    print(f"{'Board':<15} {'Backtrack Calls':<20} {'Failures':<15}")
    print(f"{'-'*50}")
    
    for board, (bt, fail) in results.items():
        print(f"{board:<15} {bt:<20} {fail:<15}")


Solving: easy.txt
Initial board:
5 3 . | . 7 . | . . . 
6 . . | 1 9 5 | . . . 
. 9 8 | . . . | . 6 . 
---------------------
8 . . | . 6 . | . . 3 
4 . . | 8 . 3 | . . 1 
7 . . | . 2 . | . . 6 
---------------------
. 6 . | . . . | 2 8 . 
. . . | 4 1 9 | . . 5 
. . . | . 8 . | . 7 9 

Solution found:
5 3 4 | 6 7 8 | 9 1 2 
6 7 2 | 1 9 5 | 3 4 8 
1 9 8 | 3 4 2 | 5 6 7 
---------------------
8 5 9 | 7 6 1 | 4 2 3 
4 2 6 | 8 5 3 | 7 9 1 
7 1 3 | 9 2 4 | 8 5 6 
---------------------
9 6 1 | 5 3 7 | 2 8 4 
2 8 7 | 4 1 9 | 6 3 5 
3 4 5 | 2 8 6 | 1 7 9 

Statistics:
  Backtrack calls: 1
  Failures: 0
  Time: 0.08 ms

Solving: medium.txt
Initial board:
. . . | . . . | . 1 2 
. . . | . . . | 3 4 5 
. . . | . . 6 | 7 8 9 
---------------------
. . 7 | . . 8 | . . 4 
. . 4 | . . 1 | . . 7 
. 9 . | . . . | . 5 . 
---------------------
8 . . | . . . | . . 6 
. . 2 | . . . | . . 3 
. . . | . . . | . . 1 

Solution found:
7 3 8 | 4 5 9 | 6 1 2 
6 2 9 | 1 8 7 | 3 4 5 
4 1 5 | 2 3 6 | 7 8 9 
----------